In [1]:
# Create a complete df that can be used for training and validation of both final models (classifier and regressor)

# Workflow:

# [SETUP - run once]
# 1. Load static rasters (DEM, slope, aspect, fetch, cluster) → flatten to base_grid_df
#    (one row per 1km pixel, pixel_id as index)
# 2. Build path dictionary: {(date, variable) → filepath} for all daily rasters
# 3. Load station csv (training stations only, with cluster_id column)
# 4. Compute intensity class thresholds from station observed_prec distribution
#  *Loop through days
#     5. Load dynamic rasters for the day (ERA5-L kriged X_s, SPPs, ERA5-L
#        atmospheric vars) → flatten and join onto base_cluster_df → daily_full_grid
#     6. Add lat, lon, date as columns to daily_full_grid
#     7. Inner merge daily_full_grid with station csv on pixel_id
#        → daily_training (station pixels only, adds observed_prec column)
#     8. Assign intensity_class to daily_training from thresholds (step 4)
#     9. Append daily_full_grid → full_grid_buffer
#         Append daily_training  → training_buffer
#
#   Save full_grid_buffer  → full_grid_df_cluster_{id}.parquet  (prediction-ready)
#   Save training_buffer   → training_df_cluster_{id}.parquet   (training-ready)

In [2]:
# CONFIGURATION

import os
import glob
import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt 
import rasterio
import xarray as xr 
import rioxarray as rxr 
from scipy.spatial import cKDTree
import pyarrow as pa
import pyarrow.parquet as pq

# Paths
resampled_rasters = r"..\Data\Processed\Classification_Regression\Temp_Rasters"
static_dir = r"..\Data\Processed\Classification_Regression\static_df"     #path to store df for static variables
daily_dynamic_train_dir = r"..\Data\Processed\Classification_Regression\daily_dynamic_train_df"
daily_dynamic_test_dir = r"..\Data\Processed\Classification_Regression\daily_dynamic_test_df"
yearly_parquet_train_dir = r"..\Data\Processed\Classification_Regression\yearly_parquet_train"
yearly_parquet_test_dir = r"..\Data\Processed\Classification_Regression\yearly_parquet_test" # Fixed
output_dir = r"..\Data\Processed\Classification_Regression"

# Ensure all directories exist
for d in [static_dir, daily_dynamic_train_dir, daily_dynamic_test_dir, 
          yearly_parquet_train_dir, yearly_parquet_test_dir, output_dir]:
    os.makedirs(d, exist_ok=True)
    
#template wgs84
wgs84_template_raster = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")


#helper and dictionary to get seasons
seasons = {
    'winter':       [12, 1, 2],
    'pre_monsoon':  [3, 4, 5],
    'monsoon':      [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}

def get_season(date_str):
    month = pd.to_datetime(date_str).month
    for season, months in seasons.items():
        if month in months:
            return season


In [3]:
# 1. Prepare df for static raster

cluster_rxr = rxr.open_rasterio(r"..\Data\Processed\Spatial_clustering\Precipitation_clusters.tif")

dem_elev_mean_rxr = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_mean.tif")
dem_elev_std_rxr = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_std.tif")
dem_elev_max_rxr = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")
dem_elev_min_rxr = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_min.tif")
dem_slope_mean_rxr = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_slope_mean.tif")
dem_aspect_mean_rxr = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_aspect_mean.tif")
dem_smoothed_rxr = rxr.open_rasterio(r"..\Data\Processed\Fetch\Preprocessing\dem_smoothed.tif")

fetch_1000_dist_rxr = rxr.open_rasterio(r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_1000m.tif")
fetch_1000_dir_rxr = rxr.open_rasterio(r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_1000m.tif")
fetch_500_dist_rxr = rxr.open_rasterio(r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_500m.tif")
fetch_500_dir_rxr = rxr.open_rasterio(r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_500m.tif")
local_relief_rxr = rxr.open_rasterio(r"..\Data\Processed\Fetch\Preprocessing\local_relief.tif")

#static dict
static_dict = {
        "cluster": cluster_rxr,
        'elev_mean': dem_elev_mean_rxr,
        'elev_std': dem_elev_std_rxr,
        'elev_max': dem_elev_max_rxr,
        'elev_min': dem_elev_min_rxr,
        'slope_mean': dem_slope_mean_rxr,
        'aspect_mean': dem_aspect_mean_rxr,
        'elev_smoothed': dem_smoothed_rxr,
        'fetch_1000m_dist': fetch_1000_dist_rxr,
        'fetch_1000m_dir': fetch_1000_dir_rxr,
        'fetch_500m_dist': fetch_500_dist_rxr,
        'fetch_500m_dir': fetch_500_dir_rxr,
        'local_relief': local_relief_rxr
    }

aligned_static_rasters = []
for name, raster in static_dict.items():
    raster = raster.rio.write_crs("EPSG: 4326")
    reproj = raster.rio.reproject_match(wgs84_template_raster)
    reproj.name = name
    reproj = reproj.squeeze('band', drop = True)
    aligned_static_rasters.append(reproj)

stacked_static_xr = xr.merge(aligned_static_rasters, compat = 'override')
stacked_static_df = stacked_static_xr.to_dataframe().reset_index()
stacked_static_df.to_csv(os.path.join(static_dir, "static_df.csv"), index = False )

del static_dict, cluster_rxr, dem_elev_mean_rxr, dem_elev_std_rxr, dem_elev_max_rxr, dem_elev_min_rxr
del dem_slope_mean_rxr, dem_aspect_mean_rxr, dem_smoothed_rxr, fetch_1000_dist_rxr
del fetch_1000_dir_rxr, fetch_500_dist_rxr, fetch_500_dir_rxr, local_relief_rxr
del aligned_static_rasters, stacked_static_xr

In [4]:
# 3 & 4. Load station csv (training stations only & get intensity threshold)

insitu_total_df = pd.read_csv(r"..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv")
train_stations_df = pd.read_csv(r"..\Data\Processed\Split_stations\train_stations.csv")
test_stations_df = pd.read_csv(r"..\Data\Processed\Split_stations\test_stations.csv")

train_index_list = list(train_stations_df['index_no'])
test_index_list = list(test_stations_df['index_no'])
insitu_train_df = insitu_total_df[insitu_total_df['index_no'].isin(train_index_list)].copy()
insitu_test_df = insitu_total_df[insitu_total_df['index_no'].isin(test_index_list)].copy()

df_insitu_rainfall_wet_days = insitu_total_df[~(insitu_total_df['rainfall_sum'] < 0.1)]
light_rain_threshold = df_insitu_rainfall_wet_days['rainfall_sum'].quantile(0.50)
print("Threshold separating light and heavy rainfall:", light_rain_threshold, "mm")

insitu_train_df['date'] = pd.to_datetime(insitu_train_df[['year', 'month', 'day']])
insitu_train_df = insitu_train_df.rename(columns={'lat': 'y', 'long': 'x', 'rainfall_sum': 'insitu_prec'})
insitu_train_df = insitu_train_df[['y',	'x', 'date', 'insitu_prec']]

insitu_test_df['date'] = pd.to_datetime(insitu_test_df[['year', 'month', 'day']])
insitu_test_df = insitu_test_df.rename(columns={'lat': 'y', 'long': 'x', 'rainfall_sum': 'insitu_prec'})
insitu_test_df = insitu_test_df[['y',	'x', 'date', 'insitu_prec']]

#print(insitu_train_df)
del insitu_total_df, train_stations_df, train_index_list, df_insitu_rainfall_wet_days

Threshold separating light and heavy rainfall: 8.4 mm


In [5]:
# 5. *Loop through days
#    Load dynamic rasters for the day (ERA5-L kriged X_s, SPPs, ERA5-L, atmospheric vars)
from tqdm import tqdm
date_range = pd.date_range(start='2001-01-01', end='2005-12-31', freq='D')

for single_date in tqdm(date_range, desc="Processing daily rasters"):
    date_str = single_date.strftime('%Y%m%d')
    
    paths = {
        "era5L_prec_resampled": f"..\\Data\\Processed\\Classification_Regression\\Temp_Rasters\\ERA5L_resampled\\ERA5L_prec_{date_str}_resampled.tif",
        "era5L_prec_kriged": f"..\\Data\\Processed\\Kriging_Predictors\\Kriged_rasters\\kriged_{date_str}.tif",
        "har_prec": f"..\\Data\\Processed\\Classification_Regression\\Temp_Rasters\\HAR_resampled\\HAR_prec_{date_str}_resampled.tif",
        "persiann_prec": f"..\\Data\\Processed\\Classification_Regression\\Temp_Rasters\\PERSIANN_resampled\\PERSIANN_prec_{date_str}_resampled.tif",
        "chirps_prec": f"..\\Data\\Processed\\Classification_Regression\\Temp_Rasters\\CHIRPS_resampled\\CHIRPS_prec_{date_str}_resampled.tif",
        "modis_lst_d": f"..\\Data\\Datasets\\MODIS_2_lst_d\\MODIS_lst_d_{date_str}.tif",
        "modis_lst_n": f"..\\Data\\Datasets\\MODIS_2_lst_n\\MODIS_lst_n_{date_str}.tif",
        "modis_lst_diff": f"..\\Data\\Datasets\\MODIS_2_lst_diff\\MODIS_lst_diff_{date_str}.tif",
        "era5L_wind_speed": f"..\\Data\\Processed\\Classification_Regression\\Temp_Rasters\\ERA5L_wind_speed_resampled\\ERA5L_wind_speed_{date_str}_resampled.tif",
        "era5L_total_evaporation": f"..\\Data\\Processed\\Classification_Regression\\Temp_Rasters\\ERA5L_total_evaporation_resampled\\ERA5L_total_evaporation_{date_str}_resampled.tif",
        "era5L_surface_pressure": f"..\Data\Processed\Classification_Regression\Temp_Rasters\ERA5L_surface_pressure_resampled\ERA5L_surface_pressure_{date_str}_resampled.tif",
        "era5L_soil_moisture": f"..\\Data\\Processed\\Classification_Regression\\Temp_Rasters\\ERA5L_soil_moisture_resampled\\ERA5L_soil_moisture_{date_str}_resampled.tif",
        "era5L_dewpoint_temperature": f"..\\Data\\Processed\\Classification_Regression\\Temp_Rasters\\ERA5L_dewpoint_temperature_resampled\\ERA5L_dewpoint_temperature_{date_str}_resampled.tif",
        "modis_evi": f"..\\Data\\Processed\\Classification_Regression\\Temp_Rasters\\MODIS_EVI_resampled\\MODIS_EVI_{date_str}_resampled.tif"
    }

    # Skip if any files are missing
    if not all(os.path.exists(p) for p in paths.values()):
        continue

    # Open and align rasters
    aligned_rasters = []
    for name, path in paths.items():
        ds = rxr.open_rasterio(path).rio.write_crs("EPSG:4326")
        reproj = ds.rio.reproject_match(wgs84_template_raster).squeeze('band', drop=True)
        reproj.name = name
        aligned_rasters.append(reproj)
    
    # Merge and create daily dataframe
    stacked_df = xr.merge(aligned_rasters, compat='override').to_dataframe().reset_index()
    daily_merged_df = stacked_df.merge(stacked_static_df, on=['x', 'y'], how='outer')
    daily_merged_df['date'] = single_date
    daily_merged_df['season'] = daily_merged_df['date'].apply(get_season)
    daily_merged_df[['x', 'y']] = daily_merged_df[['x', 'y']].round(5)

    # --- 5. cKDTree In-situ Merge ---
    insitu_train_day = insitu_train_df[insitu_train_df['date'] == single_date].copy()
    insitu_test_day = insitu_test_df[insitu_test_df['date'] == single_date].copy()

    if not insitu_train_day.empty or not insitu_test_day.empty:
        
        tree = cKDTree(daily_merged_df[['x', 'y']].values)
        if not insitu_train_day.empty:
            dist_train, idx_train = tree.query(insitu_train_day[['x', 'y']].values, k=1)
            mask_train = dist_train <= 0.006
            valid_insitu_train = insitu_train_day.iloc[mask_train].reset_index(drop=True)
            valid_train_raster = daily_merged_df.iloc[idx_train[mask_train]].reset_index(drop=True)
            
            final_df_train = pd.concat([
                valid_insitu_train, 
                valid_train_raster.drop(columns=['x', 'y', 'date'])
            ], axis=1)

            if not final_df_train.empty:
                final_df_train['intensity_class'] = np.where(
                    final_df_train['insitu_prec'] >= light_rain_threshold, 1, 0
                )
                final_df_train.to_csv(os.path.join(daily_dynamic_train_dir, f"merged_{date_str}.csv"), index=False)

        if not insitu_test_day.empty:
            dist_test, idx_test = tree.query(insitu_test_day[['x', 'y']].values, k=1)
            mask_test = dist_test <= 0.006
            
            valid_insitu_test = insitu_test_day.iloc[mask_test].reset_index(drop=True)
            valid_test_raster = daily_merged_df.iloc[idx_test[mask_test]].reset_index(drop=True)
            
            final_df_test = pd.concat([
                valid_insitu_test, 
                valid_test_raster.drop(columns=['x', 'y', 'date'])
            ], axis=1)

            if not final_df_test.empty:
                final_df_test['intensity_class'] = np.where(
                    final_df_test['insitu_prec'] >= light_rain_threshold, 1, 0
                )
                final_df_test.to_csv(os.path.join(daily_dynamic_test_dir, f"merged_{date_str}.csv"), index=False)

Processing daily rasters: 100%|██████████| 1826/1826 [57:45<00:00,  1.90s/it] 


In [6]:
#save a combined df from yearly parquet files - train df
parquet_files = []
for year in range(2001, 2006):
    writer = None
    yearly_parquet_path = os.path.join(yearly_parquet_train_dir, f"full_{year}.parquet")
    daily_files = sorted(glob.glob(os.path.join(daily_dynamic_train_dir, f"merged_{year}*.csv")))

    for filepath in daily_files:
        df_chunk = pd.read_csv(filepath)
        table = pa.Table.from_pandas(df_chunk)
        
        if writer is None:
            writer = pq.ParquetWriter(yearly_parquet_path, table.schema)
        
        writer.write_table(table)
    
    if writer:
        writer.close()
        
    df_year = pd.read_parquet(yearly_parquet_path)
    df_year.to_csv(os.path.join(yearly_parquet_train_dir, f"combined_{year}.csv"), index=False)
    
    parquet_files.append(yearly_parquet_path)
    print(f"Year {year} processed: Parquet and CSV saved.")

all_years_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
final_csv_path = os.path.join(output_dir, "final_train_df.csv")
all_years_df.to_csv(final_csv_path, index=False)

Year 2001 processed: Parquet and CSV saved.
Year 2002 processed: Parquet and CSV saved.
Year 2003 processed: Parquet and CSV saved.
Year 2004 processed: Parquet and CSV saved.
Year 2005 processed: Parquet and CSV saved.


In [7]:
#save a combined df from yearly parquet files - test df
parquet_files = []
for year in range(2001, 2006):
    writer = None
    yearly_parquet_path = os.path.join(yearly_parquet_test_dir, f"full_{year}.parquet")
    daily_files = sorted(glob.glob(os.path.join(daily_dynamic_test_dir, f"merged_{year}*.csv")))

    for filepath in daily_files:
        df_chunk = pd.read_csv(filepath)
        table = pa.Table.from_pandas(df_chunk)
        
        if writer is None:
            writer = pq.ParquetWriter(yearly_parquet_path, table.schema)
        
        writer.write_table(table)
    
    if writer:
        writer.close()
        
    df_year = pd.read_parquet(yearly_parquet_path)
    df_year.to_csv(os.path.join(yearly_parquet_test_dir, f"combined_{year}.csv"), index=False)
    
    parquet_files.append(yearly_parquet_path)
    print(f"Year {year} processed: Parquet and CSV saved.")

all_years_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
final_csv_path = os.path.join(output_dir, "final_test_df.csv")
all_years_df.to_csv(final_csv_path, index=False)

Year 2001 processed: Parquet and CSV saved.
Year 2002 processed: Parquet and CSV saved.
Year 2003 processed: Parquet and CSV saved.
Year 2004 processed: Parquet and CSV saved.
Year 2005 processed: Parquet and CSV saved.
